In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [15]:
local_dataset = pd.read_csv("data/local/cebu.csv")
local_dataset.head()

,Patient,Age,Height (cm),Weight (kg),Body Mass Index (BMI),Blood Pressure,Menstrual Irregularities,Symptoms,Ultrasound Findings,Gravida
0,P1,34,1.57,64.0,25.9645,120/80,Irregular,"irregular masses, weight gain, hirsutism",PCO,G0P0
1,P2,22,1.53,52.0,22.2137,120/80,Irregular,"irregular masses, acne, weight gain",PCO,G0P0
2,P3,19,1.55,55.0,22.8928,110/70,Regular,"irregular masses, acne",PCO,G0P0
3,P4,24,1.55,55.0,22.8928,110/70,Irregular,"irregular masses, acne, weight gain",PCO,G0P0
4,P5,25,1.59,50.0,19.7777,110/71,Regular,irregular masses,PCO,G0P0


In [16]:
global_dataset = pd.read_csv("data/global/dataset_1.csv")
global_dataset.head()

,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm)
0,1,1,0,28,44.6,152.0,19.300000,15,78,22,...,0,1.0,0,110,80,3,3,18.0,18.0,8.5
1,2,2,0,36,65.0,161.5,24.921163,15,74,20,...,0,0.0,0,120,70,3,5,15.0,14.0,3.7
2,3,3,1,33,68.8,165.0,25.270891,11,72,18,...,1,1.0,0,120,80,13,15,18.0,20.0,10.0
3,4,4,0,37,65.0,148.0,29.674945,13,72,20,...,0,0.0,0,120,70,2,2,15.0,14.0,7.5
4,5,5,0,25,52.0,161.0,20.060954,11,72,18,...,0,0.0,0,120,80,3,4,16.0,14.0,7.0


## Preprocessing

In [17]:
from adapter import Dataset1Adapter

In [18]:
adapter = Dataset1Adapter(global_dataset)
preprocessed_global_dataset = adapter.convert()

In [19]:
preprocessed_global_dataset.head()

,BMI,Age (yrs),PCOS (Y/N),hair growth(Y/N),Pimples(Y/N),Pregnant(Y/N),Cycle(R/I)
0,19.300000,28,0,0,0,0,2
1,24.921163,36,0,0,0,1,2
2,25.270891,33,1,0,1,1,2
3,29.674945,37,0,0,0,0,2
4,20.060954,25,0,0,0,1,2


In [20]:
preprocessed_global_dataset.dtypes

BMI                 float64
Age (yrs)             int64
PCOS (Y/N)            int64
hair growth(Y/N)      int64
Pimples(Y/N)          int64
Pregnant(Y/N)         int64
Cycle(R/I)            int64
dtype: object

## Modelling and Analysis

In [21]:
# Define features and target variable
X = preprocessed_global_dataset.drop(columns=['PCOS (Y/N)'])
y = preprocessed_global_dataset['PCOS (Y/N)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Random Forest

In [22]:
# global dataset has been preprocessed and is ready for analysis.
from sklearn.ensemble import RandomForestClassifier

In [23]:

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

In [24]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))    

Confusion Matrix:
[[63 14]
 [10 22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.82      0.84        77
           1       0.61      0.69      0.65        32

    accuracy                           0.78       109
   macro avg       0.74      0.75      0.74       109
weighted avg       0.79      0.78      0.78       109



### XGBoost

In [25]:
import xgboost as xgb

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100,      # Number of gradient boosted trees
    learning_rate=0.1,     # Step size shrinkage used to prevent overfitting
    max_depth=3,           # Maximum depth of a tree
    random_state=42,       # Seed for reproducibility
    eval_metric='logloss'  # Evaluation metric for validation data
)

print("Training the XGBoost model...")
model.fit(X_train, y_train)

print("Making predictions...")
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Training the XGBoost model...
Making predictions...

Model Accuracy: 80.73%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.88      0.87        77
           1       0.69      0.62      0.66        32

    accuracy                           0.81       109
   macro avg       0.77      0.75      0.76       109
weighted avg       0.80      0.81      0.80       109



# Conclusion